# 🇵🇰 Awam Assist — Citizen Service Navigator
### RAG Pipeline Notebook

---

**Project Overview**

Awam Assist is a Retrieval-Augmented Generation (RAG) based chatbot designed to help Pakistani citizens navigate government services. It answers questions about Zakat, NADRA, BISP/Ehsaas, IESCO utilities, transport, emergency services, civil registration, and police/FIR processes — in plain English or Roman Urdu.

**Tech Stack**
| Component | Tool |
|-----------|------|
| Document Loading | LangChain TextLoader / PyPDFLoader |
| Text Splitting | RecursiveCharacterTextSplitter |
| Embeddings | HuggingFace `all-MiniLM-L6-v2` |
| Vector Store | ChromaDB |
| LLM | Groq — LLaMA 3.3 70B |
| Orchestration | LangChain LCEL |
| Deployment | FastAPI + Railway |

**Knowledge Base Categories**
1. Zakat Punjab Programs
2. IESCO Electricity Services
3. Punjab Transport Services
4. ICT Civil Registration (Marriage & Birth)
5. NADRA Services
6. BISP / Ehsaas Programs
7. Rescue & Emergency Services
8. Police & FIR Process

---

## Step 1 — Install Dependencies

Install all required libraries. The `-q` flag suppresses verbose output.

> **Note:** If running on a fresh Colab session, restart the runtime after installation (`Runtime → Restart session`) before proceeding.

In [1]:
!pip install langchain langchain-community langchain-groq langchain-text-splitters chromadb sentence-transformers pypdf -q

## Step 2 — Import Libraries

Import all necessary modules for the RAG pipeline:
- **Document Loaders** — read `.txt` and `.pdf` files from the knowledge base
- **Text Splitter** — chunk documents into smaller pieces for embedding
- **Embeddings** — convert text chunks into vector representations
- **Vector Store** — store and retrieve embeddings using ChromaDB
- **LLM** — Groq-hosted LLaMA 3.3 70B for answer generation
- **Chain Components** — LangChain LCEL for building the RAG pipeline

In [2]:
import os
import zipfile

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✅ All imports successful!")

✅ All imports successful!


## Step 3 — Load Knowledge Base Documents

Extract the zip file and recursively load all `.txt` and `.pdf` files from the knowledge base.

**Knowledge Base Structure:**
```
RAG - Chatbot Citizen Service Navigator/
├── Zakat/
├── IESCO utility/
├── Transport services/
├── ICT marriage + birth certificate/
├── Nadra/
├── Ehsas and BISP Program/
├── Rescue and emergency/
└── Poilice and FIR/
```

> **Note:** PNG/image files are automatically skipped — only `.txt` and `.pdf` are loaded.

In [5]:
# ── Configuration ──────────────────────────────────────────────
ZIP_PATH     = "/content/RAG - Chatbot Citizen Service Navigator.zip"
EXTRACT_PATH = "/content/RAG - Chatbot Citizen Service Navigator"
# ───────────────────────────────────────────────────────────────

# Extract zip only if not already extracted
if not os.path.exists(EXTRACT_PATH):
    print(f"📦 Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("✅ Extraction complete.")
else:
    print("📁 Archive already extracted, skipping.")

# Recursively load all .txt and .pdf files
docs = []
skipped = []

for root, dirs, files in os.walk(EXTRACT_PATH):
    for file in files:
        file_path = os.path.join(root, file)

        if file.endswith(".txt"):
            loader = TextLoader(file_path, encoding="utf-8")
            docs.extend(loader.load())
            print(f"  ✅ Loaded: {file}")

        elif file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            docs.extend(loader.load())
            print(f"  ✅ Loaded (PDF): {file}")

        else:
            skipped.append(file)

print(f"\n📄 Total documents loaded : {len(docs)}")
print(f"⏭️  Files skipped          : {len(skipped)} (images/unsupported)")

📦 Extracting /content/RAG - Chatbot Citizen Service Navigator.zip ...
✅ Extraction complete.
  ✅ Loaded: ICT Civil Registration Services.txt
  ✅ Loaded: Nadra complete.txt
  ✅ Loaded: BISP_Ehsaas_Complete_Guide.txt
  ✅ Loaded: Emergency serice.txt
  ✅ Loaded: Iesco citizen guide.txt
  ✅ Loaded: Police_FIR_Complete_Guide.txt
  ✅ Loaded: Punjab transport complete.txt
  ✅ Loaded: Zakat punjab complete.txt

📄 Total documents loaded : 8
⏭️  Files skipped          : 0 (images/unsupported)


## Step 4 — Chunk Documents

Split documents into smaller chunks for effective retrieval.

**Parameters:**
- `chunk_size = 500` — each chunk contains ~500 characters (roughly a paragraph)
- `chunk_overlap = 50` — 50-character overlap between chunks to preserve context at boundaries

> Smaller chunks = more precise retrieval. Larger chunks = more context per retrieval. 500 is a good balance for Q&A tasks.

In [6]:
# Initialize the text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split all loaded documents into chunks
chunks = splitter.split_documents(docs)

print(f"✅ Total chunks created : {len(chunks)}")
print(f"📊 Avg chunk size       : ~{sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

✅ Total chunks created : 142
📊 Avg chunk size       : ~370 characters


## Step 5 — Generate Embeddings & Build Vector Store

Convert text chunks into numerical vector representations and store them in ChromaDB.

**Embedding Model:** `all-MiniLM-L6-v2`
- Lightweight 80MB model
- 384-dimensional embeddings
- Good balance of speed and quality for English text
- Runs locally — no API key required

**ChromaDB** stores the vectors locally at `/content/chroma_db` so they can be reused without re-embedding.

> ⏱️ This step takes 1–2 minutes on first run as it downloads the embedding model.

In [7]:
# Load the HuggingFace embedding model (downloads on first run)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Create ChromaDB vector store from document chunks
# persist_directory saves the DB to disk so it survives session restarts
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)

print(f"✅ Vector DB ready!")
print(f"📦 Stored at: /content/chroma_db")
print(f"🔢 Total vectors: {vectordb._collection.count()}")

/tmp/ipykernel_22100/2320224618.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector DB ready!
📦 Stored at: /content/chroma_db
🔢 Total vectors: 142


## Step 6 — Initialize the LLM

Connect to Groq's API to use the LLaMA 3.3 70B model for answer generation.

**Why Groq?**
- Free tier available
- Extremely fast inference (LPU hardware)
- LLaMA 3.3 70B is a high-quality open-source model

> 🔑 Get your free API key at [console.groq.com](https://console.groq.com) and replace the value below.

In [8]:
# ── Configuration ──────────────────────────────────────────────
GROQ_API_KEY = "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"   # Replace with your key
MODEL_NAME   = "llama-3.3-70b-versatile"  # Fast, high-quality open model
# ───────────────────────────────────────────────────────────────

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=MODEL_NAME
)

print(f"✅ LLM ready: {MODEL_NAME}")

✅ LLM ready: llama-3.3-70b-versatile


## Step 8 — Build the RAG Chain

Assemble the full Retrieval-Augmented Generation pipeline using LangChain's LCEL (LangChain Expression Language).

**Pipeline Flow:**
```
User Question
     ↓
Retriever — searches ChromaDB for top-3 relevant chunks
     ↓
Prompt Template — formats question + retrieved context
     ↓
LLM (Groq LLaMA 3.3 70B) — generates answer from context
     ↓
Output Parser — extracts clean string response
     ↓
Answer
```

**Retriever:** `k=3` means the top 3 most semantically similar chunks are retrieved for each query.

**System Prompt:** Instructs the LLM to:
- Answer only from retrieved context
- Use plain language (no jargon)
- Respond in the user's language (English or Roman Urdu)

In [9]:
# System prompt — defines chatbot behaviour and language style
SYSTEM_PROMPT = """
You are a helpful citizen service assistant for Pakistan.
Answer the question based only on the context provided below.
Use simple, plain language that any citizen can understand — avoid legal or technical jargon.
If the user writes in Roman Urdu, reply in Roman Urdu.
If the user writes in English, reply in English.
If the answer is not in the context, say: "Mujhe is baray mein maloomat nahi. Please relevant department se rabta karein."
Keep answers concise and practical.

Context:
{context}

Question: {question}
"""

# Build prompt template
prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)

# Configure retriever — fetch top-3 relevant chunks from ChromaDB
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Assemble the RAG chain using LCEL pipe syntax
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain assembled and ready!")

✅ RAG chain assembled and ready!


## Step 9 — Test the Pipeline

Run a single test query to verify the full RAG pipeline is working end-to-end.

In [10]:
# Single query test
query = "Am I eligible for Zakat Guzara allowance?"

print(f"❓ Question: {query}")
print(f"\n💬 Answer:")
print(chain.invoke(query))

❓ Question: Am I eligible for Zakat Guzara allowance?

💬 Answer:
Apko Zakat Guzara allowance mil sakta hai agar ap Muslim hain, Pakistan ka citizen hain, aur garibi ki laker ke neeche rahte hain. Apko apni eligibility ke liye District Zakat & Ushr Committee se rabta karna chahiye.


## Step 10 — Evaluation: Batch Test Across All Categories

Test the chatbot across all 8 knowledge base categories to verify coverage and response quality.

In [11]:
# Test questions — one per knowledge base category
test_questions = [
    # Zakat
    ("Zakat",       "What is the monthly Guzara allowance amount from Zakat?"),
    # IESCO
    ("IESCO",       "How do I apply for a new electricity connection from IESCO?"),
    # Transport
    ("Transport",   "What is the T-Cash card and how do I get one?"),
    # Civil Registration
    ("Civil Reg",   "What documents are needed to register a marriage in Islamabad?"),
    # NADRA
    ("NADRA",       "How do I renew my CNIC and how much does it cost?"),
    # BISP
    ("BISP",        "How do I check if I am eligible for BISP Kafaalat program?"),
    # Emergency
    ("Emergency",   "What is the emergency helpline number in Punjab?"),
    # Police / FIR
    ("Police/FIR",  "What should I do if police refuse to register my FIR?"),
]

print("=" * 70)
print("  AWAM ASSIST — BATCH EVALUATION")
print("=" * 70)

for category, question in test_questions:
    print(f"\n📂 Category : {category}")
    print(f"❓ Question : {question}")
    print(f"💬 Answer   : {chain.invoke(question)}")
    print("-" * 70)

  AWAM ASSIST — BATCH EVALUATION

📂 Category : Zakat
❓ Question : What is the monthly Guzara allowance amount from Zakat?
💬 Answer   : Zakat ka monthly Guzara allowance amount PKR 2,000 per month hai.
----------------------------------------------------------------------

📂 Category : IESCO
❓ Question : How do I apply for a new electricity connection from IESCO?
💬 Answer   : Aap IESCO ki website par online apply kar sakte hain, telephone ya mail ke zariye bhi apply kar sakte hain, ya phir apne nearest IESCO sub-division office ja kar bhi apply kar sakte hain.
----------------------------------------------------------------------

📂 Category : Transport
❓ Question : What is the T-Cash card and how do I get one?
💬 Answer   : T-Cash Card ek rechargeable smart banking card hai jo Punjab ke public transport services mein cashless travel ke liye use hoti hai. Iske liye aapko nearest Metro ya Orange Line station jaana hoga ya Bank of Punjab (BOP) ke designated booths par jaana hoga.
---------

## Step 11 — Interactive Chat (Optional)

A simple interactive widget to chat with the bot directly inside the notebook.

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# UI components
text_input = widgets.Text(
    placeholder="Ask about any Pakistani government service...",
    description="Your Question:",
    layout=widgets.Layout(width="650px")
)

output_area = widgets.Output()

ask_button = widgets.Button(
    description="Ask",
    button_style="primary",
    layout=widgets.Layout(width="100px")
)

# Button click handler
def on_ask(b):
    with output_area:
        clear_output()
        if text_input.value.strip():
            print("⏳ Thinking...")
            answer = chain.invoke(text_input.value)
            clear_output()
            print(f"❓ {text_input.value}")
            print(f"\n💬 {answer}")
        else:
            print("Please enter a question.")

ask_button.on_click(on_ask)

print("🇵🇰 Awam Assist — Interactive Chat")
print("Ask about: Zakat, NADRA, BISP, IESCO, Transport, Emergency, Marriage/Birth, FIR")
display(text_input, ask_button, output_area)

🇵🇰 Awam Assist — Interactive Chat
Ask about: Zakat, NADRA, BISP, IESCO, Transport, Emergency, Marriage/Birth, FIR


Text(value='', description='Your Question:', layout=Layout(width='650px'), placeholder='Ask about any Pakistan…

Button(button_style='primary', description='Ask', layout=Layout(width='100px'), style=ButtonStyle())

Output()

## Step 12 — Export ChromaDB for Deployment

Download the ChromaDB vector store to deploy with the FastAPI backend on Railway.

After downloading, replace the ChromaDB files in your Railway GitHub repo and push — Railway will auto-redeploy.

In [13]:
import shutil
from google.colab import files

# Zip the ChromaDB directory
CHROMA_DIR   = "/content/chroma_db"
OUTPUT_ZIP   = "/content/chroma_db_export"

print("📦 Zipping ChromaDB...")
shutil.make_archive(OUTPUT_ZIP, 'zip', CHROMA_DIR)
print(f"✅ Saved to {OUTPUT_ZIP}.zip")

# Download to local machine
print("⬇️  Starting download...")
files.download(f"{OUTPUT_ZIP}.zip")

📦 Zipping ChromaDB...
✅ Saved to /content/chroma_db_export.zip
⬇️  Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                    AWAM ASSIST STACK                    │
├─────────────────────────────────────────────────────────┤
│  Frontend  │  React / HTML+JS  │  Vercel / Static Host  │
│            │         ↓         │                        │
│  Backend   │  FastAPI (Python) │  Railway               │
│            │         ↓         │                        │
│  RAG Core  │  LangChain LCEL   │  ChromaDB (local)      │
│            │         ↓         │                        │
│  LLM       │  Groq API         │  LLaMA 3.3 70B         │
└─────────────────────────────────────────────────────────┘
```

## Links
- **Live API:** https://awam-assist-production.up.railway.app
- **GitHub:** _(add your repo link here)_
- **Groq Console:** https://console.groq.com

---
*Built by Murtaza Majid — Awam Assist, 2026*